# k-Nearest Neighbors (k-NN) Model

Train and evaluate a k-Nearest Neighbors classifier, find the optimal value of k, and evaluate performance.

In [ ]:
# Check if running in Google Colab and set up environment cleanly
import os
import shutil
from pathlib import Path

if 'COLAB_RELEASE_TAG' in os.environ:
    # 1. Force change directory back to the root '/content'
    os.chdir('/content')
    
    # 2. Clean up any accidental nested clone directories to save space and resolve path confusion
    nested_path = Path('/content/Bird-Species-Classification-ML/Bird-Species-Classification-ML')
    if nested_path.exists():
        print("Cleaning up accidental nested git clone folders...")
        shutil.rmtree(nested_path, ignore_errors=True)
        
    # 3. Clone repository if it doesn't exist under /content
    if not os.path.exists('Bird-Species-Classification-ML'):
        print("Cloning Bird-Species-Classification-ML repository...")
        !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
        
    # 4. Change working directory to '/content/Bird-Species-Classification-ML'
    %cd /content/Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Load pre-computed features and labels
import os
import numpy as np
import joblib
from pathlib import Path

DATA_DIR = Path("./processed_data")
npz_file = DATA_DIR / "combined_hog_color_lbp.npz"

if not npz_file.exists():
    print("Feature file not found. Trying single feature matrix fallback...")
    npz_file = DATA_DIR / "hog_features.npz"

if not npz_file.exists():
    raise FileNotFoundError("Processed feature file not found. Please run notebooks/02_Feature_Extraction.ipynb first!")

data = np.load(npz_file)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

label_mapping = joblib.load(DATA_DIR / "label_mapping.pkl")
class_names = [k.split('.')[-1].replace('_', ' ') for k in sorted(label_mapping, key=label_mapping.get)]

print(f"Loaded feature dataset: {npz_file.name}")
print(f"  Training set  : {X_train.shape}")
print(f"  Validation set: {X_val.shape}")
print(f"  Testing set   : {X_test.shape}")
print(f"  Classes ({len(class_names)}): {class_names[:5]}...")

## 1. Train Baseline k-NN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("--- Training Baseline k-NN (k=5) ---")
knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train, y_train)

y_pred_base = knn_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_base)
print(f"Baseline k-NN (k=5) Test Accuracy: {baseline_acc * 100:.2f}%")

## 2. Find the Best Value of k & Hyperparameter Tuning

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

print("--- Finding Best Value of k with GridSearchCV ---")
param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search_knn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_knn.fit(X_train, y_train)

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters : {grid_search_knn.best_params_}")
print(f"Best Cross-Val Score : {grid_search_knn.best_score_ * 100:.2f}%")

## 3. Visualize Accuracy vs. Value of k

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
scores_uniform = []
scores_distance = []

results = grid_search_knn.cv_results_
for k in k_values:
    for params, mean_score in zip(results['params'], results['mean_test_score']):
        if params['n_neighbors'] == k and params['metric'] == 'euclidean':
            if params['weights'] == 'uniform':
                scores_uniform.append(mean_score * 100)
            elif params['weights'] == 'distance':
                scores_distance.append(mean_score * 100)

plt.figure(figsize=(10, 5))
plt.plot(k_values, scores_uniform[:len(k_values)], marker='o', label='Uniform Weights (Euclidean)')
plt.plot(k_values, scores_distance[:len(k_values)], marker='s', label='Distance Weights (Euclidean)')
plt.title('k-NN Performance: Cross-Validation Accuracy vs k')
plt.xlabel('Value of k (n_neighbors)')
plt.ylabel('CV Accuracy (%)')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Evaluate & Save Best k-NN Model

In [ ]:
import seaborn as sns

best_knn = grid_search_knn.best_estimator_
y_pred = best_knn.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
print(f"Final Tuned k-NN Test Accuracy: {final_acc * 100:.2f}%")

# Save model
MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "knn_model.pkl"
joblib.dump(best_knn, model_path)
print(f"Best k-NN model saved to '{model_path}' successfully!")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Tuned k-NN')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()